# Phase 9 — model training and evaluation notebook

Trains and compares three approaches on the grouped split produced by Phase 8:

1. median-price baseline,
2. linear regression,
3. random forest.

The test split is only used for the final comparison. MAE and RMSE are in
maloti (LSL). Requires `data/processed/laptop_prices_clean.csv`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from luct_ai import evaluate, modeling

cleaned = pd.read_csv(ROOT / 'data' / 'processed' / 'laptop_prices_clean.csv', dtype='string')
len(cleaned), cleaned['split'].value_counts().to_dict()

## 1. Train and tune (test set untouched)

In [ ]:
result = modeling.train_models(cleaned)
print('best model:', result.best_model_name)
result.comparison_val

## 2. Model comparison on the held-out test set

MAE = average error in maloti; RMSE = penalises large errors; R² = share of
price variation explained.

In [ ]:
result.comparison_test

## 3. Where the model performs poorly — error by price band

In [ ]:
best_pred = result.test_predictions[result.best_model_name]
band_table = evaluate.error_by_band(result.test_truth, best_pred, result.test_bands)
band_table

In [ ]:
evaluate.actual_vs_predicted_plot(result.test_truth, best_pred,
                                  title=f'Actual vs predicted — {result.best_model_name}')

In [ ]:
evaluate.error_by_band_plot(band_table)

## 4. Is more data still helping? (learning curve)

In [ ]:
train_only = cleaned[modeling.usable_mask(cleaned) & (cleaned['split'] != 'test')]
X = modeling.feature_frame(train_only)
y = pd.to_numeric(train_only[modeling.TARGET_COLUMN], errors='coerce')

curve = evaluate.learning_curve_data(result.best_model, X, y)
print('empty?', curve.is_empty)
evaluate.learning_curve_plot(curve)

## 5. What drives the prediction? (random-forest importance)

In [ ]:
if result.feature_importances is not None:
    display(result.feature_importances.head(15))
    evaluate.feature_importance_plot(result.feature_importances)
else:
    print('random forest was not the selected model')

## 6. Save the best model + version manifest

In [ ]:
paths = modeling.save_model_bundle(result, ROOT / 'models', cleaned=cleaned)
paths